# Xu-Net with different Database Partition

## Libraries

In [ ]:
from scipy import misc, ndimage, signal
from sklearn.model_selection  import train_test_split
import numpy
import numpy as np
import random
import ntpath
import os
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from keras import optimizers 
from keras import regularizers
import tensorflow as tf
import cv2
from keras import backend as K
from time import time
import time as tm
import datetime
from operator import itemgetter
import glob
from skimage.util.shape import view_as_blocks
from keras.utils import np_utils
from keras.utils import to_categorical

## 30 SRM filters for preprocessing and the activation function

In [ ]:
################################################## 30 SRM FILTERS
srm_weights = np.load('SRM_Kernels.npy') 
biasSRM=numpy.ones(30)
print (srm_weights.shape)
################################################## TLU ACTIVATION FUNCTION
T3 = 3;
def Tanh3(x):
    tanh3 = K.tanh(x)*T3
    return tanh3
##################################################

## TPU

In [ ]:
#https://www.tensorflow.org/guide/tpu
#https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/guide/tpu.ipynb
#https://colab.research.google.com/notebooks/tpu.ipynb#scrollTo=_pQCOmISAQBu
%tensorflow_version 2.x
import tensorflow as tf
print("Tensorflow version " + tf.__version__)

try:
  tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # TPU detection
  print('Running on TPU ', tpu.cluster_spec().as_dict()['worker'])
except ValueError:
  raise BaseException('ERROR: Not connected to a TPU runtime; please see the previous cell in this notebook for instructions!')

tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)
tpu_strategy = tf.distribute.TPUStrategy(tpu)

## Xu-Net architecture

In [ ]:
def Xu_Net( img_size=256, compile=True):
    
    #tf.reset_default_graph()
    #tf.keras.backend.clear_session()
    print ("using",2,"classes")
    
    #Preprocessing
    inputs = tf.keras.Input(shape=(img_size,img_size,1), name="input_1")
    layers = tf.keras.layers.Conv2D(30, (5,5), weights=[srm_weights,biasSRM], strides=(1,1), trainable=False, activation=Tanh3, use_bias=True)(inputs)


    
    #Block 1
    
    #Layer 0
    layers = Conv2D(8, (5,5), strides=(1,1),padding="same", kernel_initializer='glorot_normal', kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers) 
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = Lambda(tf.keras.backend.abs)(layers)
    layers = BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=True, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = Concatenate()([layers, layers, layers])
    
    #Block 2
    
    #Layer 1
    layers = tf.keras.layers.SpatialDropout2D(rate=0.1)(layers)
    layers = Conv2D(16, (5,5), strides=1,padding="same", kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers) 
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=True, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)  
    layers = AveragePooling2D((5,5), strides= 2, padding='same')(layers)
    
    #Block 3
    
    #Layer 2
    layers = tf.keras.layers.SpatialDropout2D(rate=0.1)(layers)
    layers = Conv2D(32, (1,1), strides=1,padding="same", kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers) 
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = AveragePooling2D((5,5), strides= 2,padding="same")(layers)
    
    #Block 4
    #Layer 3
    layers = tf.keras.layers.SpatialDropout2D(rate=0.1)(layers)
    layers = Conv2D(64, (1,1), strides=1,padding="same", kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers) 
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = AveragePooling2D((5,5), strides=2,padding="same")(layers)
    #Block 5
    #Layer 4
    layers = tf.keras.layers.SpatialDropout2D(rate=0.1)(layers)
    layers = Conv2D(128, (1,1), strides=1,padding="same", kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers)
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = Concatenate()([layers, layers, layers])
    layers = GlobalAveragePooling2D(data_format="channels_last")(layers)
    
    #Block 6
    #Layer 5, FC, Softmax
  
    #FC
    layers = Dense(128,kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers)
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = Dense(64,kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers)
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
    layers = Dense(32,kernel_initializer='glorot_normal',kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers)
    layers = ReLU(negative_slope=0.1, threshold=0)(layers)
   
    #Softmax
    predictions = Dense(2, activation="softmax", name="output_1",kernel_regularizer=tf.keras.regularizers.l2(0.0001),bias_regularizer=tf.keras.regularizers.l2(0.0001))(layers)
    model =tf.keras.Model(inputs = inputs, outputs=predictions)
    #Compile
    optimizer = tf.keras.optimizers.SGD(learning_rate=0.005, momentum=0.95)
    
    if compile:
        model.compile(optimizer= optimizer,
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
        print ("Xunet")
    return model

## Defining different functions to work with the architecture

In [ ]:

def Final_Results_Valid(PATH_trained_models):
    global AccValid
    global LossValid
    AccValid = []
    LossValid = [] 
    B_accuracy = 0 #B --> Best
    for filename in sorted(os.listdir(PATH_trained_models)):
        if filename != ('train') and filename != ('validation'):
            print(filename)
            with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
                 _model = Xu_Net()
            _model.load_weights(PATH_trained_models+'/'+filename)
            loss,accuracy = _model.evaluate(X_valid, y_valid, verbose=0)
            print(f'Loss={loss:.4f} y Accuracy={accuracy:0.4f}'+'\n')

            BandAccValid  = accuracy
            BandLossValid = loss
            AccValid.append(BandAccValid)    
            LossValid.append(BandLossValid)  
            
            if accuracy > B_accuracy:
                B_accuracy = accuracy
                B_loss = loss
                B_name = filename
    
    print("\n\nBest")
    print(B_name)
    print(f'Loss={B_loss:.4f} y Accuracy={B_accuracy:0.4f}'+'\n')

In [ ]:
def Final_Results_Train(PATH_trained_models):
    global AccTrain
    global LossTrain
    AccTrain = []
    LossTrain = [] 
    B_accuracy = 0 #B --> Best
    for filename in sorted(os.listdir(PATH_trained_models)):
        if filename != ('train') and filename != ('validation'):
            print(filename)
            with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
                 _model = Xu_Net()
            _model.load_weights(PATH_trained_models+'/'+filename)
            loss,accuracy = _model.evaluate(X_train, y_train, verbose=0)
            print(f'Loss={loss:.4f} y Accuracy={accuracy:0.4f}'+'\n')

            BandAccTrain  = accuracy
            BandLossTrain = loss
            AccTrain.append(BandAccTrain)    
            LossTrain.append(BandLossTrain)  
            
            if accuracy > B_accuracy:
                B_accuracy = accuracy
                B_loss = loss
                B_name = filename
    
    print("\n\nBest")
    print(B_name)
    print(f'Loss={B_loss:.4f} y Accuracy={B_accuracy:0.4f}'+'\n')

In [ ]:
def Final_Results_Test(PATH_trained_models):
    global AccTest
    global LossTest
    AccTest = []
    LossTest= [] 
    B_accuracy = 0 #B --> Best
    for filename in sorted(os.listdir(PATH_trained_models)):
        if filename != ('train') and filename != ('validation'):
            print(filename)
            with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
                 _model = Xu_Net()
            _model.load_weights(PATH_trained_models+'/'+filename)
            loss,accuracy = _model.evaluate(X_test, y_test, verbose=0)
            print(f'Loss={loss:.4f} y Accuracy={accuracy:0.4f}'+'\n')

            BandAccTest  = accuracy
            BandLossTest = loss
            AccTest.append(BandAccTest)    
            LossTest.append(BandLossTest)  
            
            if accuracy > B_accuracy:
                B_accuracy = accuracy
                B_loss = loss
                B_name = filename
    
    print("\n\nBest")
    print(B_name)
    print(f'Loss={B_loss:.4f} y Accuracy={B_accuracy:0.4f}'+'\n')

In [ ]:
def graphics(AccTest, AccTrain, AccValid, LossTest, LossTrain, LossValid, model_name, path_img_base):
    if not os.path.exists(path_img_base+"/"+model_name):
       os.makedirs(path_img_base+"/"+model_name)

    with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
        model = Xu_Net()
    
    lossTEST,accuracyTEST   = model.evaluate(X_test, y_test,verbose=None)
    lossTRAIN,accuracyTRAIN = model.evaluate(X_train, y_train,verbose=None)
    lossVALID,accuracyVALID = model.evaluate(X_valid, y_valid,verbose=None)

    with plt.style.context('seaborn-white'):
        plt.figure(figsize=(10, 10))
        plt.plot(np.concatenate([np.array([accuracyTRAIN]),np.array(AccTrain)],axis=0))
        plt.plot(np.concatenate([np.array([accuracyVALID]),np.array(AccValid)],axis=0))
        plt.plot(np.concatenate([np.array([accuracyTEST]),np.array(AccTest)],axis=0)) #Test
        plt.title('Accuracy Vs Epoch')
        plt.ylabel('Accuracy')
        plt.xlabel('Epoch')
        plt.legend(['Train', 'Validation', 'Test'], loc='upper left')
        plt.grid('on')
        plt.savefig(path_img_base+'/'+model_name+'/Accuracy_XU_Net_'+model_name+'.eps', format='eps')
        plt.savefig(path_img_base+'/'+model_name+'/Accuracy_XU_Net_'+model_name+'.svg', format='svg')
        plt.savefig(path_img_base+'/'+model_name+'/Accuracy_XU_Net_'+model_name+'.pdf', format='pdf')     
        plt.show()
        
        plt.figure(figsize=(10, 10))
        plt.plot(np.concatenate([np.array([lossTRAIN]),np.array(LossTrain)],axis=0))
        plt.plot(np.concatenate([np.array([lossVALID]),np.array(LossValid)],axis=0))
        plt.plot(np.concatenate([np.array([lossTEST]),np.array(LossTest)],axis=0)) #Test
        plt.title('Loss Vs Epoch')
        plt.ylabel('Loss')
        plt.xlabel('Epoch')
        plt.legend(['Train', 'Validation', 'Test'], loc='upper left')
        plt.grid('on')
        plt.savefig(path_img_base+'/'+model_name+'/Loss_XU_Net_'+model_name+'.eps', format='eps')
        plt.savefig(path_img_base+'/'+model_name+'/Loss_XU_Net_'+model_name+'.svg', format='svg')
        plt.savefig(path_img_base+'/'+model_name+'/Loss_XU_Net_'+model_name+'.pdf', format='pdf') 
        plt.show()

In [ ]:
def top_models(AccTest,AccTrain,AccValid):
    numbers=AccTest
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% total epochs
        index, value = numbers_sort[i]
        print("Test Accuracy {}, epoch:{}\n".format(value, index+1))
    
    print("")
    
    numbers=AccTrain
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% total epochs
        index, value = numbers_sort[i]
        print("Train Accuracy {}, epoch:{}\n".format(value, index+1))
    
    print("")
    
    numbers=AccValid
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% total epochs
        index, value = numbers_sort[i]
        print("Validation Accuracy {}, epoch:{}\n".format(value, index+1))

In [ ]:
def trainTPU(path_model, epochs, model_Name):
    global model_name
    start_time = tm.time()
    model_name = model_Name
    path_log_base = path_model+'/'+model_Name
    if not os.path.exists(path_log_base):
        os.makedirs(path_log_base)

    with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
         model = Xu_Net()

    epoch_ = 1
    for epoch in range(epochs):
        epoch=epoch+1
        print("epoch ",epoch)
        model.fit(X_train,y_train,validation_data=(X_valid,y_valid), batch_size=128*2, epochs=epoch_, verbose=1) 
        model.save_weights(path_model+'/'+model_name+'/'+str(epoch).zfill(4)+'.hdf5', overwrite=True) 

    TIME = tm.time() - start_time
    print("Time "+model_name+" = %s [seconds]" % TIME)

## Working with BOSSbase 1.01 WOW y PAYLOAD = 0.4bpp

In the README, there is a link to download the databases we use for the work. There is BOSSbase 1.01, cover images and stego. The steganographic algorithms used in the paper are WOW and S-UNIWARD, with a payload of 0.4bpp.

Note: If you want to train the algorithm with S-UNIWARD 0.4 bpp, change "PATH04_WOW1" and  "base_name".

# Database Partition

Choose any of the Database partition and train model. 

Note: Any of the nine "CNN Input" can be used to train your model, you can also download the databases in PGM format through a link found in the README.

## Training image pairs(4000), Validation image pairs(1000), and Test image pairs(5000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.50
VAL_RATIO   = 0.20   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## Training image pairs(2500), Validation image pairs(2500), and Test image pairs(5000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.50
VAL_RATIO   = 0.50   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## Training image pairs(4000), Validation image pairs(3000), and Test image pairs(3000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.30
VAL_RATIO   = 0.43   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## Training image pairs(8000), Validation image pairs(1000), and Test image pairs(1000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.10
VAL_RATIO   = 0.111   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## CNN name and algorithm 

## Training

In [ ]:
path_model = "./WOW/logs"
path_img_base = "./Image/WOW/images"

model_Name = "Xu-Net..."

trainTPU(path_model=path_model, epochs=150, model_Name = "Xu-Net...")

## Test

In [ ]:
Final_Results_Test(path_model+"/"+model_Name) 

## Validation

In [ ]:
Final_Results_Valid(path_model+"/"+model_Name) 

## Train

In [ ]:
Final_Results_Train(path_model+"/"+model_Name) 

## Training, validation and testing graph

In [ ]:
graphics(AccTest, AccTrain, AccValid, LossTest, LossTrain, LossValid, model_Name, path_img_base)

## Top

In [ ]:
top_models(AccTest,AccTrain,AccValid)

Note: If you want to train the algorithm with S-UNIWARD 0.4 bpp, change "PATH04_WOW1" and  "base_name".